# **ZERO-SHOT CLASSIFICATION FOR LLAMA-3.2-1B**


In [4]:
# load test dataset, retrieve this from the .csv file

import pandas as pd

test_frame = pd.read_csv('fpb_test.csv')

text_col = 'text'  # text
label_col = 'label' # label
source_col = 'source' # sure why not
unique_labels = test_frame[label_col].unique().tolist()

print(f"loaded {len(test_frame)} rows.")
print(f"labels to test: {unique_labels}")

display(test_frame)

loaded 727 rows.
labels to test: ['Neutral', 'Fear', 'Optimism', 'Sadness', 'Joy']


,text,label,source
0,the share capital of alma media corporation bu...,Neutral,FinancialPhraseBank
1,the eu commission said earlier it had fined th...,Fear,FinancialPhraseBank
2,"kesko pursues a strategy of healthy , focused ...",Optimism,FinancialPhraseBank
3,down to eur5 .9 m h1 '09 3 august 2009 - finni...,Sadness,FinancialPhraseBank
4,"cencorp would focus on the development , manuf...",Neutral,FinancialPhraseBank
...,...,...,...
722,7 march 2011 - finnish it company digia oyj he...,Optimism,FinancialPhraseBank
723,"operating loss totaled eur 0.8 mn , compared t...",Sadness,FinancialPhraseBank
724,aspocomp has repaid its interest bearing liabi...,Optimism,FinancialPhraseBank
725,the contract also includes cutting and edging ...,Neutral,FinancialPhraseBank


In [5]:
# log into huggingface

import os
import sys

google_colab = "google.colab" in sys.modules and not os.environ.get("VERTEX_PRODUCT")

if google_colab:
    # Use secret if running in Google Colab
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
else:
    # Store Hugging Face data under `/content` if running in Colab Enterprise
    if os.environ.get("VERTEX_PRODUCT") == "COLAB_ENTERPRISE":
        os.environ["HF_HOME"] = "/content/hf"
    # Authenticate with Hugging Face
    from huggingface_hub import get_token
    if get_token() is None:
        from huggingface_hub import notebook_login
        notebook_login()

**TO-DO 01: Make a detailed DataFrame with these:
Text / Correct Label / Prediction Label / Is Correct**

**Purpose:** we look at which labels were the most correctly predicted, we can create a second dataframe evaluating the accuracy score of certain labels

**TO-DO 02: Compare these with our models?**

In [6]:
!pip install -q transformers accelerate bitsandbytes scikit-learn

import torch
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# configure model
model_id = "meta-llama/Llama-3.2-1B"

# load model and tokeniser
tokenizer = AutoTokenizer.from_pretrained(model_id)

# load in 4-bit to standard colab gpu memory
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    quantization_config=quantization_config
)

# classify function for zero-shot classification. we don't need any correct samples, just load and go
def classify(text, candidate_labels):
    prompt = f"Classify the following text into one of these categories: {', '.join(candidate_labels)}.\n\nText: {text}\nCategory:"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_new_tokens=10, do_sample=False)
    response = tokenizer.decode(output[0], skip_special_tokens=True)
    prediction = response.split("Category:")[-1].strip().lower()

    # Find the best match in candidate labels
    for label in candidate_labels:
        if label.lower() in prediction:
            return label
    return "unknown" # did something go wrong?

# time to evaluate!
y_true = test_frame[label_col].tolist() # all the correct labels
y_pred = [classify(text, unique_labels) for text in test_frame[text_col]] # all the predicted labels

# basic metrics
accuracy = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted')

# export metrics
results_dict = {
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score'],
    'Value': [accuracy, precision, recall, f1]
}

# create dataframe
results_df = pd.DataFrame(results_dict)

# export to csv
results_df.to_csv('zero_shot_llama_01.csv', index=False)

print("Results saved to 'zero_shot_llama_01.csv'")
display(results_df)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.5 MB/s eta 0:00:00


config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for op

Results saved to 'zero_shot_gemma_01.csv'


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


,Metric,Value
0,Accuracy,0.581843
1,Precision,0.355525
2,Recall,0.581843
3,F1 Score,0.441363


**IMPORTANT: FIX "pad_token_id" AND "UndefinedMetricWarning" ERRORS!!!**

